In [16]:
import os
import re

from dotenv import load_dotenv
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

# Azure OpenAI setup
load_dotenv()

openai_endpoint = os.getenv("ENDPOINT")
model_deployment = os.getenv("MODEL_DEPLOYMENT")

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

client = OpenAI(
    base_url=openai_endpoint,
    api_key=token_provider()
)

# ReAct Agent
class Agent:

    def __init__(self, system=""):
        self.messages = []

        if system:
            self.messages.append(
                {
                    "role": "system",
                    "content": system
                }
            )

    def __call__(self, message):

        self.messages.append(
            {
                "role": "user",
                "content": message
            }
        )

        completion = client.chat.completions.create(
            model=model_deployment,
            # temperature=0,
            messages=self.messages
        )

        result = completion.choices[0].message.content

        self.messages.append(
            {
                "role": "assistant",
                "content": result
            }
        )

        return result


prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.

At the end of the loop output an Answer.

Available actions:

calculate
average_dog_weight

Examples:

Action: average_dog_weight: Border Collie

Action: calculate: 37 + 20
"""

# Tools
def calculate(expression):
    return str(eval(expression))


def average_dog_weight(name):

    weights = {
        "Border Collie": 37,
        "Scottish Terrier": 20,
        "Toy Poodle": 7,
        "Bulldog": 51
    }

    return f"{name} average weight is {weights.get(name, 50)} lbs"


known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

action_re = re.compile(r"Action:\s*(\w+):\s*(.*)")


def query(question, max_turns=10):

    bot = Agent(prompt)

    next_prompt = question

    for _ in range(max_turns):

        result = bot(next_prompt)

        print("\n====================")
        print(result)
        print("====================")

        match = None

        for line in result.splitlines():
            m = action_re.search(line)
            if m:
                match = m
                break

        if not match:
            return result

        action, action_input = match.groups()

        print(f"\nRunning Tool -> {action}({action_input})")

        # execute tool
        tool_func = known_actions.get(action)

        if tool_func is None:
            raise Exception(f"Unknown action: {action}")

        observation = tool_func(action_input)

        print("Observation:", observation)

        next_prompt = f"Observation: {observation}"


question = """
I have 2 dogs, a Border Collie and a Scottish Terrier.
What is their combined weight?
"""

query(question)


Thought: Get the average weight for a Border Collie.
Action: average_dog_weight: Border Collie
PAUSE

Running Tool -> average_dog_weight(Border Collie)
Observation: Border Collie average weight is 37 lbs

Thought: Get the average weight for a Scottish Terrier.
Action: average_dog_weight: Scottish Terrier
PAUSE

Running Tool -> average_dog_weight(Scottish Terrier)
Observation: Scottish Terrier average weight is 20 lbs

Thought: Add the two average weights together to get combined weight.
Action: calculate: 37 + 20
PAUSE

Running Tool -> calculate(37 + 20)
Observation: 57

Answer: Their combined average weight is about 57 lbs.


'Answer: Their combined average weight is about 57 lbs.'